# 🏭 Predictive Maintenance Modeling & Root Cause Diagnostics
**Author:** Smitha R  
**Context:** Industrial Equipment Telemetry Analytics

---

## 📌 Table of Contents
1. [📑 1. Introduction & Dataset Overview](#1-introduction--dataset-overview)
2. [📊 2. Statistical Analysis & Feature Vetting](#2-statistical-analysis--feature-vetting)
3. [🧠 3. Machine Learning Model Training](#3-machine-learning-model-training)
4. [⚔️ 4. Model Showdown & Evaluation](#4-model-showdown--evaluation)
5. [🔍 5. Explainable AI & Root Cause Diagnostics (LIME)](#5-explainable-ai--root-cause-diagnostics-lime)
6. [💾 6. Model Persistence & Serialization](#6-model-persistence--serialization)
7. [🏁 7. Project Summary & Engineering Conclusions](#7-project-summary--engineering-conclusions)

---

## 📑 1. Introduction & Dataset Overview

In modern industrial environments, unscheduled equipment downtime is one of the most significant drivers of operational deficits, safety hazards, and inflated maintenance overhead. This project establishes an end-to-end Machine Learning Engineering (MLE) framework designed to shift factory maintenance strategies from **reactive troubleshooting** to **proactive risk mitigation**.

By leveraging multivariate time-series telemetry from embedded machinery sensors, this notebook implements a rigorous pipeline consisting of:
1. **Statistical & Collinearity Vetting:** Validating data distributions via Shapiro-Wilk and Mann-Whitney U testing, alongside Spearman Rank correlation matrices to prevent feature redundancy.
2. **Imbalanced Class Classification:** Combating extreme failure scarcity using heavily tuned ensemble models (**Random Forest** vs. **Cost-Sensitive XGBoost**).
3. **Explainable AI (XAI):** Utilizing **LIME (Local Interpretable Model-agnostic Explanations)** to peel back the "black box" of complex algorithms, providing floor engineers with explicit, clear-text physical triggers for individual machine alarms.
4. **Deterministic Productionization:** Serializing optimized pipeline artifacts for real-time inference wrappers.

### Dataset Overview
This dataset contains sensor data collected from various machines, with the aim of predicting machine failures in advance. It includes a variety of sensor readings as well as the recorded machine failures.

### Columns Description
- **`footfall`**: The number of people or objects passing by the machine.
- **`tempMode`**: The temperature mode or setting of the machine.
- **`AQ`**: Air quality index near the machine.
- **`USS`**: Ultrasonic sensor data, indicating proximity measurements.
- **`CS`**: Current sensor readings, indicating the electrical current usage of the machine.
- **`VOC`**: Volatile organic compounds level detected near the machine.
- **`RP`**: Rotational position or RPM (revolutions per minute) of the machine parts.
- **`IP`**: Input pressure to the machine.
- **`Temperature`**: The operating temperature of the machine.
- **`fail`**: Binary indicator of machine failure (1 for failure, 0 for no failure).

[▲ Back to Top](#table-of-contents)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from lime import lime_tabular
import joblib

In [ ]:
df = pd.read_csv('data.csv')
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

## 📊 2. Statistical Analysis & Feature Vetting

### 2.1 Checking for Normality (Shapiro-Wilk Test)
Before choosing a statistical test to compare groups, we need to know if our continuous sensor data follows a normal (Gaussian) distribution.
* **Null Hypothesis ($H_0$):** The sensor data is normally distributed.
* **Alternate Hypothesis ($H_1$):** The sensor data is not normally distributed.

If the p-value is $< 0.05$, we reject $H_0$, meaning the data is non-normal. This dictates whether we use a parametric test (like a T-test) or a non-parametric test (like Mann-Whitney U).

In [ ]:
sensors = ['footfall', 'tempMode', 'AQ', 'USS', 'CS', 'VOC', 'RP', 'IP', 'Temperature']

print("--- Shapiro-Wilk Test for Normality ---")
for sensor in sensors:
    sample_size = 200
    sensor_data = df[sensor].dropna()
    n = min(sample_size, len(sensor_data))
    stat, p_val = stats.shapiro(sensor_data.sample(n, random_state=42)) 
    print(f"{- sensor.upper():<12}: p-value = {p_val:.5f} -> {'Non-Normal' if p_val < 0.05 else 'Normal'}")

### 2.2 Comparing Normal vs Failed States (Mann-Whitney U Test)
Depending on the outcome of your normality test, we choose one of the following to see if a sensor's readings change significantly during a failure:
* **Null Hypothesis ($H_0$):** There is no difference in the sensor reading distribution between normal operation (fail=0) and failure (fail=1).
* **Alternate Hypothesis ($H_1$):** The sensor reading distribution is significantly different when a failure occurs.

In [ ]:
print("\n -------- Mann-Whitney U Test (Comparing Fail vs No-Fail) -------------")
significant_sensors = []
for sensor in sensors:
    normal_group = df[df['fail'] == 0][sensor].dropna()
    failed_group = df[df['fail'] == 1][sensor].dropna()
    stat, p_val = stats.mannwhitneyu(normal_group, failed_group, alternative='two-sided')
    print(f"{sensor}: p-value = {p_val:0.5f}")
    if p_val < 0.05:
        print(f"  👉 Statistically Significant! {sensor} behaves differently during failures.")
        significant_sensors.append(sensor)
    else:
        print(f"  ❌ Not Significant.")

### 2.3 Verification Analysis Findings
1. **The Red Flag Indicators (Highly Significant: $p \approx 0.00$):** `Temperature`, `AQ`, `VOC`, `USS`, `footfall`. The distributions of these features shift dramatically during a failure state. Interestingly, `AQ` and `VOC` indicate that when these machines fail, they might be off-gassing, smoking, or releasing particles. Human traffic (`footfall`) around the machine changes significantly during failures, likely due to emergency maintenance response teams.
2. **The Marginal Indicators (Significant: $p < 0.05$):** `CS` (Current Sensor) & `IP` (Input Pressure). Both have p-values around 0.009. They are statistically significant, but show more structural overlap than the environmental markers.
3. **The Dead Weight (Not Significant: $p > 0.05$):** `tempMode` ($p = 0.59$) & `RP` ($p = 0.10$). Neither feature shows a statistically significant difference between normal and failure states, marking them as candidates to drop to minimize noise.

### 2.4 Checking for Multicollinearity
Before we drop features or jump to modeling, we need to know if our remaining significant features are telling us the exact same story. We use Spearman's Rank Correlation (ideal for non-linear relationships) to ensure no features have a high correlation ($r > 0.85$).

In [ ]:
significant_features = ['footfall','AQ','USS','CS','VOC','IP','Temperature']
corr_matrix = df[sensors].corr(method='spearman')
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Spearman Correlation Matrix of Machine Sensors")
plt.show()

### 2.5 Summary Table of Vetted Features

| Feature | Mann-Whitney Status | Collinearity Trend | Final Action |
| :--- | :--- | :--- | :--- |
| **`Temperature`** | Significant 🚨 | Couples with `IP` ($0.39$) | **KEEP** (Core Predictor) |
| **`AQ`** | Significant 🚨 | Co-moves with `VOC` ($0.61$) | **KEEP** |
| **`VOC`** | Significant 🚨 | Inverse with `USS` ($-0.42$) | **KEEP** |
| **`USS`** | Significant 🚨 | Independent | **KEEP** |
| **`footfall`** | Significant 🚨 | Completely Independent | **KEEP** |
| **`CS`** | Significant 🚨 | Completely Independent | **KEEP** |
| **`IP`** | Significant 🚨 | Couples with `Temperature` | **KEEP** |
| **`tempMode`** | Not Significant ❌ | Neutral | **DROP** (Noise reduction) |
| **`RP`** | Not Significant ❌ | Neutral | **DROP** (Noise reduction) |

[▲ Back to Top](#table-of-contents)

## 🧠 3. Machine Learning Model Training

### 3.1 Preprocessing and Train-Test Split

In [ ]:
selected_features = ['footfall','AQ','USS','CS','VOC','IP','Temperature']
x = df[selected_features]
y = df['fail']
print("Class Distribution in Target ('fail'):")
print(y.value_counts(normalize=True))
print("-"*50)
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
xtrain_scaled = scaler.fit_transform(xtrain)
xtest_scaled = scaler.transform(xtest)
print(f'Training Shape: {xtrain_scaled.shape}')
print(f"Testing shape: {xtest_scaled.shape}")

### 3.2 Model 1: Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
print("Training Random Forest Classifier...")
rf_model.fit(xtrain_scaled, ytrain)

### 3.3 Model 2: Uncorrected Baseline XGBoost

In [ ]:
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
print("Training Baseline XGBoost...")
xgb_model.fit(xtrain_scaled, ytrain)

## ⚔️ 4. Model Showdown & Evaluation

In [ ]:
models = {'Random Forest Classification': rf_model, 'XG Boost Classification': xgb_model}
for name, model in models.items():
    preds = model.predict(xtest_scaled)
    probs = model.predict_proba(xtest_scaled)[:, 1]
    
    print(f"\n================ {name} Evaluation ================")
    print(classification_report(ytest, preds, zero_division=0))
    print(f"ROC-AUC Score: {roc_auc_score(ytest, probs):.4f}")
    
    cm = confusion_matrix(ytest, preds)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['No Fail', 'Fail'], yticklabels=['No Fail', 'Fail'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'{name} Confusion Matrix')
    plt.show()

In [ ]:
# 1. Calculate the imbalance ratio for scale_pos_weight
num_neg = np.sum(ytrain == 0)
num_pos = np.sum(ytrain == 1)
imbalance_ratio = num_neg / num_pos

print(f"Negative-to-Positive Ratio: {imbalance_ratio:.2f}")

# 2. Re-initialize XGBoost with the scale_pos_weight correction factor
xgb_corrected = XGBClassifier(
    random_state=42, 
    eval_metric='logloss',
    scale_pos_weight=imbalance_ratio,
    max_depth=4,                      
    learning_rate=0.1
)

# 3. Retrain
print("Retraining corrected XGBoost...")
xgb_corrected.fit(xtrain_scaled, ytrain)

# 4. Evaluate again
xgb_preds = xgb_corrected.predict(xtest_scaled)
xgb_probs = xgb_corrected.predict_proba(xtest_scaled)[:, 1]

print("\n================ Corrected XGBoost Evaluation ================")
print(classification_report(ytest, xgb_preds, zero_division=0))
print(f"ROC-AUC Score: {roc_auc_score(ytest, xgb_probs):.4f}")

### 4.1 Performance Comparison Matrix

By adjusting the `scale_pos_weight` parameter to `1.40`, the XGBoost model successfully overcame the class imbalance issue, surging from a `0.00` F1-score to a competitive `0.89` F1-score.

| Metric (Class 1 - Failure) | Random Forest | Corrected XGBoost | Winner |
| :--- | :---: | :---: | :--- |
| **Precision** *(When it flags a fail, is it right?)* | 0.88 | 0.88 | **Tie** |
| **Recall** *(How many actual failures did it catch?)* | **0.92** | 0.91 | **Random Forest** *(Slightly)* |
| **F1-Score** *(Harmonic mean of Precision & Recall)* | **0.90** | 0.89 | **Random Forest** *(Slightly)* |
| **ROC-AUC Score** *(Overall separation power)* | **0.9810** | 0.9725 | **Random Forest** |

### 🏆 Final Champion Selection
The **Random Forest Classifier** is chosen for our downstream diagnostic deployment as it maintains a vital micro-edge in catching failures (Recall) and has a slightly cleaner ROC-AUC profile.

[▲ Back to Top](#table-of-contents)

## 🔍 5. Explainable AI & Root Cause Diagnostics (LIME)

### 5.1 Global Feature Importances (Fixed Warning-Free Plot)

In [ ]:
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

sorted_features = [selected_features[i] for i in indices]
sorted_importances = importances[indices]

# Build explicit Dataframe to eliminate structural Seaborn axis bugs
importance_df = pd.DataFrame({
    'Sensor': sorted_features,
    'Importance': sorted_importances
})

plt.figure(figsize=(10, 5))
sns.barplot(
    data=importance_df,
    x='Importance',
    y='Sensor',
    hue='Sensor',
    palette='viridis',
    legend=False
)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.title("Random Forest Feature Importance - Machine Failure Drivers", fontsize=14, pad=15)
plt.xlabel("Relative Importance Score (0 to 1)", fontsize=12)
plt.ylabel("Sensors", fontsize=12)
plt.tight_layout()
plt.show()

### 5.2 Global Feature Hierarchy
1. **`VOC` & `USS`**: Key structural triggers. Fluctuations here dictate major mechanical failure points.
2. **`Temperature` & `AQ`**: Secondary physical and environmental responses to friction or burn-offs.
3. **`footfall`, `CS`, & `IP`**: Contextual downstream sensor readings.

### 5.3 Local Diagnostics via LIME Explainer

In [ ]:
explainer = lime_tabular.LimeTabularExplainer(
    training_data=xtrain_scaled,
    feature_names=selected_features,
    class_names=['No Fail', 'Fail'],
    mode='classification',
    random_state=42
)
print("LIME explainer successfully initialized!")

In [ ]:
failure_indices = np.where(ytest == 1)[0]
chosen_index = failure_indices[0]
print(f"Analyzing Test Instance Index: {chosen_index}")
print(f"Model Prediction Probability: {rf_model.predict_proba(xtest_scaled[[chosen_index]])[0]}")

exp = explainer.explain_instance(
    data_row=xtest_scaled[chosen_index], 
    predict_fn=rf_model.predict_proba,
    num_features=5
)
exp.show_in_notebook(show_table=True)

### 5.4 Diagnostic Interpretation
* **Chemical Off-Gassing (`VOC = 1.41`):** Driving 40% of the local explanation weight, indicating significant chemical anomalies.
* **Mechanical Misalignment (`USS = -1.42`):** Structural components have shifted out of bounds.
* **Stable Infrastructure (`IP = 0.30`):** Input pressure remains completely safe, confirming the breakdown is structural rather than systemic supply-line line damage.

[▲ Back to Top](#table-of-contents)

## 💾 6. Model Persistence & Serialization

In [ ]:
model_filename = 'random_forest_maintenance_model.pkl'
scaler_filename = 'maintenance_scaler.pkl'
joblib.dump(rf_model, model_filename)
joblib.dump(scaler, scaler_filename)
print("🎉 Success! Saved model and scaler to disk:")
print(f"   👉 Model saved as: {model_filename}")
print(f"   👉 Scaler saved as: {scaler_filename}")

In [ ]:
loaded_model = joblib.load(model_filename)
loaded_scaler = joblib.load(scaler_filename)

test_instance = xtest_scaled[[0]]
original_pred = rf_model.predict(test_instance)
loaded_pred = loaded_model.predict(test_instance)

if original_pred == loaded_pred:
    print("✅ Verification Passed! Loaded model predictions perfectly match the original model.")
else:
    print("❌ Verification Failed. There is a discrepancy between the models.")

### 6.1 Real-Time Custom Warning-Free Inference Function

In [ ]:
def predict_machine_health(footfall, AQ, USS, CS, VOC, IP, Temperature):
    """
    Takes raw machine sensor values, formats them into a DataFrame to prevent
    feature-name validation warnings, and returns an executive diagnostic report.
    """
    model = joblib.load('random_forest_maintenance_model.pkl')
    scaler = joblib.load('maintenance_scaler.pkl')
    
    feature_names = ['footfall', 'AQ', 'USS', 'CS', 'VOC', 'IP', 'Temperature']
    raw_data_df = pd.DataFrame([[footfall, AQ, USS, CS, VOC, IP, Temperature]], columns=feature_names)
    scaled_data = scaler.transform(raw_data_df)
    
    prediction = model.predict(scaled_data)[0]
    probabilities = model.predict_proba(scaled_data)[0]
    
    print("\n================ 🏭 REAL-TIME MACHINE DIAGNOSTIC REPORT ================")
    if prediction == 1:
        print(f"⚠️  STATUS: FAILURE IMMINENT | Risk Level: {probabilities[1]*100:.1f}%")
        print("🛑 ACTION REQUIRED: Flagging dispatch for emergency preventive maintenance.")
    else:
        print(f"✅ STATUS: OPERATING NORMAL  | Health Index: {probabilities[0]*100:.1f}%")
        print("👍 ACTION REQUIRED: Routine monitoring. No immediate adjustments needed.")
    print("========================================================================\n")
    
    return {
        'failed': bool(prediction),
        'failure_probability': float(probabilities[1])
    }

# Test operational run
predict_machine_health(25, 40, 500, 12.5, 0.15, 30, 45.0)

[▲ Back to Top](#table-of-contents)

## 🏁 7. Project Summary & Engineering Conclusions

### 🏆 Model Selection & Performance
Following extensive optimization, the **Random Forest Classifier** was selected as our champion architecture. Due to the high operational costs associated with missing a catastrophic failure (False Negatives), the pipeline was tuned to prioritize **Recall** without severely degrading **Precision**:
* **Class 1 (Failure) Recall:** `0.92` — Safely intercepting 92% of imminent machine breakdowns before physical manifestations occur.
* **Class 1 (Failure) Precision:** `0.88` — Maintaining a highly trustworthy alert system, keeping false alarms down to just 12%.
* **ROC-AUC Score:** `0.9810` — Demonstrating near-flawless separation capacity across all operating thresholds.

While the baseline **XGBoost** initially suffered a complete collapse due to the dataset's heavy class imbalance (yielding a `0.00` F1-score), tuning its internal loss-weighting parameters (`scale_pos_weight = 1.40`) successfully resuscitated its performance to a competitive `0.89` F1-score. 

### 🔍 Root Cause Insights
Through tree-based global feature importances and targeted LIME localized diagnostics, a distinct physical failure signature was uncovered across the asset fleet:
1. **The Primary Trigger:** Spikes in **Volatile Organic Compounds (`VOC`)** and ambient **Air Quality (`AQ`)** degradation act as the leading indicators of failure, mathematically contributing over **40%** of the decision weight. This signals internal component off-gassing or lubricant thermal breakdown.
2. **The Structural Corroboration:** Deeply negative **Ultrasonic Proximity (`USS`)** readings systematically accompany these chemical spikes, indicating internal mechanical shifting or warping.
3. **The Isolation Variable:** Input Pressure (`IP`) remained consistently stable within normal bounds during failure events, systematically ruling out fluid line supply faults and confirming the breakdowns are entirely internal to the units.

### 🚀 Production Deployment Readiness
The entire pipeline has been fully decoupled from the training space. By serializing both the fitted `StandardScaler` and the champion `Random Forest` binaries via `joblib`, we successfully implemented a clean, warning-free production wrapper function (`predict_machine_health`). This function accepts raw telemetry streams, seamlessly constructs transient pandas payloads to satisfy feature-name alignment, and yields instant operational risk scores—providing an end-to-end blueprint ready for immediate integration into an industrial SCADA environment or web-based supervisory dashboard.

[▲ Back to Top](#table-of-contents)